In [29]:
import os
from dotenv import load_dotenv
from sqlalchemy import text, create_engine
import pandas as pd
import numpy as np


In [30]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, array, array_remove, lit, array_contains, concat_ws, size, array_compact
from pyspark.ml.fpm import FPGrowth


In [31]:
# KhÃ¡Â»Å¸i Ã„â€˜Ã¡Â»â„¢ng spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FP-Growth Python")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.extraJavaOptions", "-Duser.timezone=Asia/Ho_Chi_Minh")
    .config("spark.executor.extraJavaOptions", "-Duser.timezone=Asia/Ho_Chi_Minh")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.4")
    .config("spark.sql.session.timeZone", "Asia/Ho_Chi_Minh")
    .getOrCreate()
)

spark._jvm.java.util.TimeZone.setDefault(
    spark._jvm.java.util.TimeZone.getTimeZone("Asia/Ho_Chi_Minh")
)


In [32]:
# DB config + inputs
load_dotenv("../../.env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DWH = os.getenv("DB_SCHEMA_DWH")

jdbc_url = (
    f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
    "?options=-c%20TimeZone=Asia/Ho_Chi_Minh"
)

connection_properties = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Metric source remains the EDA wide CSV; buy rule weights come only from DB.
df_metric = pd.read_csv("../EDA/outputs/fact_metric_wide_format.csv")

dim_weight_buy = pd.read_sql(
    text(f"""
        select id, rank_no, rule_col, rule_name, weight, indicator_family
        from {DWH}.dim_weight_buy
        order by rank_no
    """),
    engine
)

print("dim_weight_buy rows:", len(dim_weight_buy))
dim_weight_buy



dim_weight_buy rows: 16


,id,rank_no,rule_col,rule_name,weight,indicator_family
0,obv_lt_obv_ma_20,1,X1,obv < obv_ma_20,0.14,Volume Flow
1,rsi_14_gt_70,2,X2,rsi_14 > 70,0.12,RSI
2,bb_percent_b_20_gt_1_0,3,X3,bb_percent_b_20 > 1.0,0.11,Bollinger
3,close_price_gt_bb_upper_20,4,X4,close_price > bb_upper_20,0.11,Bollinger
4,vol_ratio_20_gt_2_and_return_1d_gt_0,5,X5,vol_ratio_20 > 2 AND return_1d > 0,0.10,Volume Spike
5,atr_14_gt_atr_ma_14,6,X6,atr_14 > atr_ma_14,0.09,Volatility
6,return_5d_gt_0_08,7,X7,return_5d > 0.08,0.09,Momentum
7,return_3d_gt_0_05,8,X8,return_3d > 0.05,0.08,Momentum
8,close_price_gte_high_10d_mul_0_98,9,X9,close_price >= high_10d * 0.98,0.08,Breakout
9,volume_gt_vol_ma_5_mul_1_5,10,X10,volume > vol_ma_5 * 1.5,0.07,Volume


In [33]:
pdf = df_metric.copy()
pdf["period_date"] = pd.to_datetime(pdf["period_date"])
pdf = pdf.sort_values(["symbol_key", "period_date"]).reset_index(drop=True)

g = pdf.groupby("symbol_key", sort=False)
prev1 = g.shift(1)
prev2 = g.shift(2)

feature_exprs = {
    "obv_lt_obv_ma_20": pdf["obv"] < pdf["obv_ma_20"],
    "rsi_14_gt_70": pdf["rsi_14"] > 70,
    "bb_percent_b_20_gt_1_0": pdf["bb_percent_b_20"] > 1.0,
    "close_price_gt_bb_upper_20": pdf["close_price"] > pdf["bb_upper_20"],
    "vol_ratio_20_gt_2_and_return_1d_gt_0": (pdf["vol_ratio_20"] > 2) & (pdf["return_1d"] > 0),
    "atr_14_gt_atr_ma_14": pdf["atr_14"] > pdf["atr_ma_14"],
    "return_5d_gt_0_08": pdf["return_5d"] > 0.08,
    "return_3d_gt_0_05": pdf["return_3d"] > 0.05,
    "close_price_gte_high_10d_mul_0_98": pdf["close_price"] >= pdf["high_10d"] * 0.98,
    "volume_gt_vol_ma_5_mul_1_5": pdf["volume"] > pdf["vol_ma_5"] * 1.5,
    "bb_width_20_increasing": pdf["bb_width_20"] > prev1["bb_width_20"],
    "rsi_14_decreasing_3_days": (pdf["rsi_14"] < prev1["rsi_14"]) & (prev1["rsi_14"] < prev2["rsi_14"]),
    "obv_decreasing_3_days": (pdf["obv"] < prev1["obv"]) & (prev1["obv"] < prev2["obv"]),
    "macd_hist_decreasing_3_days": (pdf["macd_hist"] < prev1["macd_hist"]) & (prev1["macd_hist"] < prev2["macd_hist"]),
    "return_1d_gt_0_04": pdf["return_1d"] > 0.04,
    "close_price_increasing_3_days": (pdf["close_price"] > prev1["close_price"]) & (prev1["close_price"] > prev2["close_price"]),
}

unknown_features = [rule_id for rule_id in dim_weight_buy["id"] if rule_id not in feature_exprs]
if unknown_features:
    raise ValueError(f"No feature expression defined for: {unknown_features}")

for rule in dim_weight_buy.itertuples(index=False):
    pdf[rule.rule_col] = feature_exprs[rule.id].fillna(False).astype(int)

rule_cols = dim_weight_buy["rule_col"].tolist()
weights = dim_weight_buy.set_index("rule_col").loc[rule_cols, "weight"].astype(float)

pdf["rule_weight_total"] = pdf[rule_cols].mul(weights, axis=1).sum(axis=1)
pdf["prediction"] = np.where(pdf["rule_weight_total"] >= 0.75, "sell", "buy")

pdf = pdf.dropna(subset=["return_next_3d"]).copy()
pdf["tomorrow_up"] = (pdf["return_next_3d"] > 0).astype(int)
pdf["actual_signal"] = np.where(pdf["tomorrow_up"] == 1, "buy", "sell")
pdf["actual"] = (pdf["actual_signal"] == "buy").astype(int)

model_cols = [
    "period_date",
    "symbol_key",
    "close_price",
    "rule_weight_total",
    "prediction",
    "tomorrow_up",
    "actual",
    "actual_signal",
] + rule_cols

rules_metric = pdf[model_cols].copy()
df = spark.createDataFrame(rules_metric)

print("rules:", len(dim_weight_buy))
print("rows:", df.count())
print("buy predictions:", df.filter(col("prediction") == "buy").count())
print("sell predictions:", df.filter(col("prediction") == "sell").count())




rules: 16
rows: 46552
buy predictions: 45518
sell predictions: 1034


FP-Growth buy only



In [34]:
from pyspark.ml.fpm import FPGrowth

# Only use rows predicted as buy.
df_buy = df.filter(col("prediction") == "buy")

# Real target: next-period return is positive.
df_buy = df_buy.withColumn(
    "Buy",
    when(col("actual_signal") == "buy", 1).otherwise(0)
)

item_exprs = [
    when(col(c) == 1, lit(c)).otherwise(None)
    for c in rule_cols
]

target_expr = when(col("Buy") == 1, lit("Buy")).otherwise(None)

df_items_buy = df_buy.withColumn(
    "items",
    array_compact(array(*item_exprs, target_expr))
).select("items")

count_buy = df_items_buy.count()

fp_buy = FPGrowth(
    itemsCol="items",
    minSupport=5 / count_buy,
    minConfidence=0.6
)

model_buy = fp_buy.fit(df_items_buy)
rules_buy = model_buy.associationRules



In [35]:
rule_buy = (
    rules_buy
    .filter(size(col("antecedent")) >= 3)
    .filter(array_contains(col("consequent"), "Buy"))
    .filter(~array_contains(col("antecedent"), "Buy"))
    .filter(col("confidence") >= 0.6)
)

rules_buy_out = rule_buy.select(
    concat_ws(",", col("antecedent")).alias("antecedents"),
    concat_ws(",", col("consequent")).alias("consequents"),
    col("confidence"),
    col("lift")
)



In [36]:
print("count_buy:", count_buy)
print("rules_buy:", rules_buy.count())
print("rules_buy_filtered:", rule_buy.count())
rules_buy_out.orderBy(col("confidence").desc()).show(20, False)



count_buy: 45518
rules_buy: 10076
rules_buy_filtered: 73
+--------------------------+-----------+------------------+------------------+
|antecedents               |consequents|confidence        |lift              |
+--------------------------+-----------+------------------+------------------+
|X8,X16,X9,X14,X6          |Buy        |1.0               |2.1075099546254283|
|X15,X5,X10,X6,X1,X11      |Buy        |1.0               |2.1075099546254283|
|X8,X9,X14,X6              |Buy        |1.0               |2.1075099546254283|
|X8,X9,X14,X6,X11          |Buy        |1.0               |2.1075099546254283|
|X15,X5,X10,X1,X11         |Buy        |1.0               |2.1075099546254283|
|X8,X2,X9,X14,X6           |Buy        |1.0               |2.1075099546254283|
|X8,X16,X9,X14             |Buy        |1.0               |2.1075099546254283|
|X8,X2,X16,X9,X14          |Buy        |1.0               |2.1075099546254283|
|X8,X2,X9,X14,X11          |Buy        |1.0               |2.1075099546254

In [37]:
# Run this cell after checking rules_buy_out.
with engine.begin() as conn:
    conn.execute(text(f"truncate table {DWH}.fact_cal_rules_fp_growth_buy"))

rules_buy_out.write.jdbc(
    url=jdbc_url,
    table=f"{DWH}.fact_cal_rules_fp_growth_buy",
    mode="append",
    properties=connection_properties
)

with engine.begin() as conn:
    conn.execute(text(f"call {DWH}.update_fact_cal_rules_fp_growth_prc();"))

print("appended buy rules:", rules_buy_out.count())



appended buy rules: 73
